In [ ]:
from time import sleep

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

url = "https://chicagoengineersfoundation.awardspring.com/"

review_df = {'Reviewer'                  :  [],
             'Applicant'                 :  [],
             'Community Service / Work'  :  [],
             'Short Essay'               :  [],
             'Bonus/Discretionary Points':  [],
             'Notes'                     :  []
             }

review_feedback_df = pd.DataFrame(review_df)

driver = webdriver.Firefox()

In [ ]:
review_feedback_df = pd.DataFrame(review_df)

In [ ]:
driver.get(f'{url}')

In [ ]:
# Login

email_xpath = '//*[@id="UserEmail"]'
email = 'cportiza221@gmail.com'

inputElement = driver.find_element(by=By.XPATH, value=email_xpath)
inputElement.send_keys(email)

pw_xpath = '//*[@id="Password"]'
pw = 'Cps45326551:))))'

inputElement = driver.find_element(by=By.XPATH, value=pw_xpath)
inputElement.send_keys(pw)

inputElement.send_keys(Keys.ENTER)

In [ ]:
# Go to Reviewers
url = "https://chicagoengineersfoundation.awardspring.com/Admin/Users/Reviewers"

driver.get(f'{url}')

In [ ]:
def get_reviewer_scores():
    commwork = 0
    essay = 0
    bonus = 0

    for score in range(0, 3):
        try: 
            xpath = f'//*[@id="score-value{score}"]'
            
            element = driver.find_element(by=By.XPATH, value=xpath)
        except Exception as e:
            print(f'Error occurred while finding score element {score}:{e} update xpath if necessary')
            continue

        try:
            copied_text = element.get_attribute('value')
        except Exception as e:
            print(f'Error occurred while copying text from score element {score}: {e}')
            continue
        
        if score == 0:
            commwork = int(copied_text) if copied_text else 0
        elif score == 1:
            essay = int(copied_text) if copied_text else 0
        elif score == 2:
            bonus = int(copied_text) if copied_text else 0

    try: 
        note_xpath = '/html/body/app-root/body/app-layout/admin-layout/div/section/div/main/div/reviewer/review-scoring-layout/div/div/div[2]/review-scoring-scores/div/div[2]/div/div/textarea'
        note_ele = driver.find_element(by=By.XPATH, value=note_xpath)
        note_ele.click()
        notes = note_ele.get_attribute('value') or ''
    except Exception as e:
        print(f'Error occurred while finding/clicking notes element: {e}')
        return commwork, essay, bonus, ''

    return commwork, essay, bonus, notes


In [ ]:
impersonate_user_xpath = '/html/body/section/div/main/div/div[1]/span[2]/input'
impersonate_user_xpath = '//*[@id="impersonate"]'


def get_reviewer_feedback(review_feedback_df, num_reviewers, start_pos, batch_number):
    sleep(2)
    reviewer_number = 1
    for x in range(start_pos, num_reviewers):
        url = "https://chicagoengineersfoundation.awardspring.com/Admin/Users/Reviewers"

        driver.get(f'{url}')
        sleep(2)

        # We need to move to a different page once we have finished the first page of reviewers
        if (batch_number == 2): 
            next_page_xpath = '/html/body/section/div/main/div/div[2]/ul/li[6]/a/span'
            inputElement = driver.find_element(by=By.XPATH, value=next_page_xpath)
            inputElement.click()
            sleep(2)

        re_xpath = f'/html/body/section/div/main/div/div[2]/table/tbody/tr[{x}]/td[1]/a'
        #print(re_xpath)
        try: 
            inputElement = driver.find_element(by=By.XPATH, value=re_xpath)
        except Exception as e: 
            print(f'Reviewer at position {x} is not assigned to any students')
            continue
        
        reviewer_name = inputElement.text
        inputElement.send_keys(Keys.ENTER)
        sleep(2)
        
        # Impersonate Reviewer
        try:
            inputElement = driver.find_element(by=By.XPATH, value=impersonate_user_xpath)
            inputElement.click()
            sleep(3)
        except Exception as e:
            sleep(4)
            inputElement = driver.find_element(by=By.XPATH, value=impersonate_user_xpath)
            inputElement.click()
            sleep(2)
        
        # Get the reviewer group 
        reviewer_group_path = f'/html/body/section/div/main/div/div/div/div/div/div/table/tbody/tr/td[1]/span/a'

        student_count_reviewed = 0
        student_count_not_reviewed = 0
        
        try:
            inputElement = driver.find_element(by=By.XPATH, value=reviewer_group_path)
        except Exception as e:
            print(f'Reviewer {reviewer_name} has no students to review')
            try:
                stop_impers_xpath = '//*[@id="stopImpersonate"]'
                stop_impers_element = driver.find_element(by=By.XPATH, value=stop_impers_xpath)
                stop_impers_element.click()
                sleep(3)
            except Exception as e:
                sleep(5)
                stop_impers_xpath = '//*[@id="stopImpersonate"]'
                stop_impers_element = driver.find_element(by=By.XPATH, value=stop_impers_xpath)
                stop_impers_element.click()
                sleep(2)
            continue
        
        inputElement = driver.find_element(by=By.XPATH, value=reviewer_group_path)
        inputElement.send_keys(Keys.ENTER)
        sleep(3)

        student_xpath = f'/html/body/app-root/body/app-layout/admin-layout/div/section/div/main/div/reviewer/review-group-applicants-layout/div/div[3]/table/tbody/tr[2]/td[1]'
        try:
            inputElement = driver.find_element(by=By.XPATH, value=student_xpath)
            inputElement.click()
            sleep(5)

            more_students = True
            while more_students:
                try: 
                    student_name_xpath = '/html/body/app-root/body/app-layout/admin-layout/div/section/div/main/div/reviewer/review-scoring-layout/div/div/div[2]/review-scoring-scores/div/div[2]/div/h4'
                    student_name = driver.find_element(by=By.XPATH, value=student_name_xpath).text
                except Exception as e:
                    print(f'Error occurred while finding student name: {e}, update xpath if necessary')

                commwork, essay, bonus, notes = get_reviewer_scores()

                if (commwork == 0 and essay == 0 and bonus == 0 and notes == ''):
                    print(f'Student {student_name} has not been reviewed or scored by reviewer {reviewer_name}')
                    student_count_not_reviewed += 1
                else: 
                    student_count_reviewed += 1
                    # print(student_name, commwork, essay, bonus, notes)
                    new_review_df = {'Reviewer'                 : reviewer_name,
                                    'Applicant'                 : student_name,
                                    'Community Service / Work'  : commwork,
                                    'Short Essay'               : essay,
                                    'Bonus/Discretionary Points': bonus,
                                    'Notes'                     : notes
                                    }
                    
                    review_feedback_df = pd.concat([review_feedback_df, pd.DataFrame.from_records([new_review_df])])

                try:
                    next_button_xpath = '/html/body/app-root/body/app-layout/admin-layout/div/section/div/main/div/reviewer/review-scoring-layout/div/div/div[1]/review-scoring-detail/div/div[2]/div[2]/div[1]/button[2]'
                    next_button_element = driver.find_element(by=By.XPATH, value=next_button_xpath)
                    next_button_element.send_keys(Keys.ENTER)
                    sleep(2)
                except Exception as e:
                    #print(f'No more students for reviewer {reviewer_name}')
                    more_students = False
                    
            sleep(2)
        except Exception as e:
            print(f'Error occurred while processing reviewer {reviewer_name}: {e}')

        sleep(2)

        # Stop Impersonating
        print(f"Reviewer {reviewer_number}: {reviewer_name} finished reviewing {student_count_reviewed}/{student_count_not_reviewed + student_count_reviewed} students")
        try:
            stop_impers_xpath = '//*[@id="stopImpersonate"]'
            stop_impers_element = driver.find_element(by=By.XPATH, value=stop_impers_xpath)
            stop_impers_element.click()
            sleep(2)
            reviewer_number += 1
        except Exception as e:
            sleep(5)
            stop_impers_xpath = '//*[@id="stopImpersonate"]'
            stop_impers_element = driver.find_element(by=By.XPATH, value=stop_impers_xpath)
            stop_impers_element.click()
            sleep(2)
    return review_feedback_df


In [ ]:
review_feedback_df_1 = get_reviewer_feedback(review_feedback_df, 26, 1, 1)

In [ ]:
review_feedback_df_2 = get_reviewer_feedback(review_feedback_df, 26, 1, 2)

In [ ]:
#given that I have review_feedback_df_1 and review_feedback_df_2, I want to combine them into one dataframe
eview_feedback_df = pd.concat([review_feedback_df_1, review_feedback_df_2], ignore_index=True)

In [ ]:
review_feedback_df.drop_duplicates(inplace=True)
display(review_feedback_df.head(10))
review_feedback_df.to_excel('2026 CEF Reviewer Detailed Feedback.xlsx')
review_feedback_df = pd.read_excel('2026 CEF Reviewer Detailed Feedback.xlsx', engine='openpyxl')

In [ ]:
display(review_feedback_df_1.head(25))

In [ ]:
#review_feedback_df = get_reviewer_feedback(review_feedback_df, 17, 1)

In [ ]:
#review_feedback_df = get_reviewer_feedback(review_feedback_df, 11, 1)


In [ ]:
#review_feedback_df = pd.read_excel('2025 CEF Reviewer Detailed Feedback.xlsx')
#review_feedback_second_df = get_reviewer_feedback(review_feedback_df, 11, 1)

In [ ]:
# Merge review_feedback_df and review_feedback_second_df
#review_feedback_df = pd.concat([review_feedback_df, review_feedback_second_df])
#review_feedback_df.drop_duplicates(inplace=True)
#display(review_feedback_df.head(10))
#review_feedback_df.to_excel('2025 CEF Reviewer Detailed Feedback.xlsx')

In [ ]:
# Note: NEed to start on the Reviewers Screen
#review_feedback_df = get_reviewer_feedback(review_feedback_df, 17, 1)
#for next_num in range(1, 14):
#    next_rebutton_xpath = '/html/body/section/div/main/div/div[2]/div/ul/li[6]/a/span'
#    next_rebutton_element = driver.find_element(by=By.XPATH, value=next_rebutton_xpath)
#    next_rebutton_element.click()
#    sleep(2)
#    review_feedback_df = get_reviewer_feedback(review_feedback_df, next_num + 1, start_pos=next_num)

In [ ]:
#review_feedback_df.drop_duplicates(inplace=True)
#display(review_feedback_df.head(10))
#review_feedback_df.to_excel('2025 CEF Reviewer Detailed Feedback.xlsx')

In [ ]:
import googlemaps

In [ ]:
#gmaps = googlemaps.Client(key='')

home_address = "3745 W Wilson, Chicago, IL 60625"

arrive_time = "2025-04-22T08:00:00-05:00"
#gmaps.directions(home_address, "Walter Payton College Prep", mode="driving", arrival_time=arrive_time,region="us")

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.options import Options
from time import sleep
import urllib.parse

# Set up Firefox driver (similar to your setup)
options = Options()
#options.add_argument("--headless")  # Run in headless mode to avoid opening browser window
driver = webdriver.Firefox(options=options)

In [ ]:
def get_distance_and_times(start_location, end_location):
    """
    Get distance and travel times from Google Maps for given locations.
    Loads the page once and extracts distance, driving time, and transit time.
    
    Args:
        start_location (str): Starting location name
        end_location (str): Ending location name
    
    Returns:
        dict: {'distance': str, 'driving_time': str, 'transit_time': str} or None if failed
    """
    # URL encode the locations
    start_encoded = urllib.parse.quote(start_location)
    end_encoded = urllib.parse.quote(end_location)
    
    # Construct Google Maps directions URL (defaults to driving)
    url = f"https://www.google.com/maps/dir/{start_encoded}/{end_encoded}"
    
    try:
        driver.get(url)
        sleep(5)  # Wait for page to load
        
        # Distance XPath
        distance_xpath = "/html/body/div[1]/div[2]/div[9]/div[8]/div/div/div[1]/div[2]/div/div[1]/div/div/div[5]/div[1]/div[1]/div/div[1]/div[2]/div"
        
        # Time XPaths
        driving_time_xpath = "/html/body/div[1]/div[2]/div[9]/div[3]/div[1]/div[2]/div/div[1]/div/div/div/div[2]/button/div[2]"
        transit_time_xpath = "/html/body/div[1]/div[2]/div[9]/div[3]/div[1]/div[2]/div/div[1]/div/div/div/div[3]/button/div[2]"
        
        # Extract distance
        try:
            distance_element = driver.find_element(By.XPATH, distance_xpath)
            distance = distance_element.text
        except:
            distance = "Not found"
        
        # Extract driving time
        try:
            driving_time_element = driver.find_element(By.XPATH, driving_time_xpath)
            driving_time = driving_time_element.text
        except:
            driving_time = "Not found"
        
        # Extract transit time
        try:
            transit_time_element = driver.find_element(By.XPATH, transit_time_xpath)
            transit_time = transit_time_element.text
        except:
            transit_time = "Not found"
        
        return {'distance': distance, 'driving_time': driving_time, 'transit_time': transit_time, 'url': url}
    
    except Exception as e:
        print(f"Error getting directions: {e}")
        return None

In [ ]:
# Example usage
start = "Chicago, IL"
end = "New York, NY"

# Get driving directions
info = get_distance_and_times(start, end)
if info:
    print(f"Driving - Distance: {info['distance']}, Driving Time: {info['driving_time']}, Transit Time: {info['transit_time']}")

In [ ]:
# Process all students for distance and travel times
import pandas as pd
from time import sleep

# Read the DistanceInformation.csv
distance_df = pd.read_csv('processingDataFiles/DistanceInformation.csv')

# Prepare results list
results = []

# Process each student
for index, row in distance_df.iterrows():
    student_name = f"{row['FirstName']} {row['LastName']}"
    home_address = f"{row['Home Address']}, {row['City']}"
    school_name = row['High School']
    
    print(f"Processing {student_name}...")
    
    # Get driving information
    travel_info = get_distance_and_times(home_address, school_name)
    if travel_info:
        distance = travel_info['distance']
        driving_time = travel_info['driving_time']
        transit_time = travel_info['transit_time']
    else:
        distance = "Error"
        driving_time = "Error"
        transit_time = "Error"

    # Append to results
    results.append({
        'Student Name': student_name,
        'distance': distance,
        'home_to_school_driving_time': driving_time,
        'home_to_school_transit_time': transit_time,
        'url': url
    })

    url = travel_info['url'] if travel_info else "Error"
    print(f"Finished processing {index}: {student_name}. Distance: {distance}, Driving Time: {driving_time}, Transit Time: {transit_time}")
    
    # Sleep to avoid rate limiting
    sleep(5)

In [ ]:
# Create DataFrame from results
results_df = pd.DataFrame(results)

In [ ]:
print(results_df.head())

In [ ]:
# Save to CSV
results_df.to_csv('processingDataFiles/Student_Distances_and_Times.csv', index=False)

print("Processing complete. Results saved to processingDataFiles/Student_Distances_and_Times.csv")